In [ ]:
import os
import mne
import mne_bids
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from glob import glob
from mne_bids import BIDSPath, find_matching_paths, make_report, print_dir_tree, read_raw_bids
from scipy.stats import ttest_rel

import matplotlib.patches as mpatches
%matplotlib qt
# plt.ioff()

In [ ]:
bids_root = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids"

deriv_dir_motor = os.path.join(bids_root, 'derivatives', 'cortical_filtered_motor')
if not os.path.exists(deriv_dir_motor):
    os.makedirs(deriv_dir_motor)

deriv_dir_status = os.path.join(bids_root, 'derivatives', 'cortical_filtered_status')
if not os.path.exists(deriv_dir_status):
    os.makedirs(deriv_dir_status) 

In [ ]:
task_list = ['passive']
bl_start = 0.100
l_freq = 1
h_freq = 30
passive_hex = '#4477AA'
active_hex = '#CC6677'

In [ ]:

for sub_num in range(2, 14):
    sub_label = f'{sub_num:02d}'

    # initialize an empty dictionary for data
    task_evoked_dict = {}
    event_evoked_dict = {}

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list = []

        for run_label in range(1,6):
        # load in EEG data
            bids_path = BIDSPath(root=bids_root, datatype='eeg', 
                                    subject=str(sub_label), 
                                    task=task_label, 
                                    run=run_label)
            
            try:
                data = read_raw_bids(bids_path)
                data.load_data()

                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'])
                
                # filter data
                data_filtered = data_ref.copy().filter(l_freq=l_freq, h_freq=h_freq)


                # loading events from the status events
                events_path = bids_path.copy().update(suffix='events', extension='.tsv')
                events_df = pd.read_csv(events_path.fpath, delimiter='\t')

                # dropping events that are start less than 300ms from the previous
                diffs_onset = np.diff(events_df['onset'])
                too_close = np.where(diffs_onset < 0.3)[0] + 1
                events_df = events_df.drop(index=too_close).reset_index(drop=True)

                annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.trial_type)
                data_filtered.set_annotations(annot)
                events_from_annot, event_dict2 = mne.events_from_annotations(data_filtered)
                # events_fpath = os.path.join(events_dir, f'sub-{sub_label}', f'sub-{sub_label}_task-{task_label}_run-{run_label}_events.tsv')
                # events_df = pd.read_csv(events_fpath, delimiter='\t') 
                # annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.type)
                # data_filtered.set_annotations(annot)
                # events_from_annot, event_dict2 = mne.events_from_annotations(data_filtered)

                
                # epoch data based on stimulus events
                epochs = mne.Epochs(data_filtered, 
                                    events=events_from_annot, #events, 
                                    event_id=event_dict2, #event_dict,
                                    on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-1*bl_start, tmax=0.4, 
                                    baseline=[-1*bl_start, 0],
                                    reject=dict(eeg=75e-6)).drop_bad()
                
                epoch_list.append(epochs)

            except Exception as e:
                print(f"No run {run_label} for task-{task_label}: {e}")
        
        try:
            # combine epochs across runs
            all_epochs = mne.concatenate_epochs(epoch_list)
        except IndexError as e:
            print(f'cannot run for sub-{sub_label} task-{task_label} run-{run_label}: {e}')

        try:
            out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif'
            all_epochs.save(os.path.join(deriv_dir, out_base), overwrite=False)
        except Exception as e:
            print(f"Error saving epochs for sub-{sub_label} task-{task_label}: {e}")
            continue


In [ ]:

for sub_num in range(2, 35):

    if sub_num == 8:
        continue
    sub_label = f'{sub_num:02d}'

    # initialize an empty dictionary for data
    task_evoked_dict = {}
    event_evoked_dict = {}

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list = []

        for run_label in range(1,6):
        # load in EEG data
            bids_path = BIDSPath(root=bids_root, datatype='eeg', 
                                    subject=str(sub_label), 
                                    task=task_label, 
                                    run=run_label)
            
            try:
                data = read_raw_bids(bids_path)
                data.load_data()

                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'])
                
                # filter data
                data_filtered = data_ref.copy().filter(l_freq=l_freq, h_freq=h_freq)


                # loading events from the status channel
                events_path = bids_path.copy().update(suffix='events', extension='.tsv')
                events_df = pd.read_csv(events_path.fpath, delimiter='\t')
                # dropping events that are start less than 300ms from the previous
                diffs_onset = np.diff(events_df['onset'])
                too_close = np.where(diffs_onset < 0.3)[0] + 1
                events_df = events_df.drop(index=too_close).reset_index(drop=True)
                annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.trial_type)
                data_filtered.set_annotations(annot)
                events_from_annot, event_dict2 = mne.events_from_annotations(data_filtered)
                
                # epoch data based on stimulus events
                epochs = mne.Epochs(data_filtered, 
                                    events=events_from_annot, #events, 
                                    event_id=event_dict2, #event_dict,
                                    on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-1*bl_start, tmax=0.4, 
                                    baseline=[-1*bl_start, 0],
                                    reject=dict(eeg=75e-6)).drop_bad()
                
                epoch_list.append(epochs)

            except Exception as e:
                print(f"No run {run_label} for task-{task_label}: {e}")
        
        try:
            # combine epochs across runs
            all_epochs = mne.concatenate_epochs(epoch_list)
        except IndexError as e:
            print(f'cannot run for sub-{sub_label} task-{task_label} run-{run_label}: {e}')
        except Exception as e:
            print(f"Exception occured: {e}")
            print()

        try:
            out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'
            # if task_label == 'motor':
            #     all_epochs.save(os.path.join(deriv_dir_motor, out_base), overwrite=True)
            # else:
            #     all_epochs.save(os.path.join(deriv_dir_status, out_base), overwrite=True)
        except Exception as e:
            print(f"Error saving epochs for sub-{sub_label} task-{task_label}: {e}")
            continue


In [ ]:
epoch_list

In [ ]:
act_pos_avgs = []
act_neg_avgs = []
act_eps = sorted(glob(deriv_dir_status+f'/sub-{sub_num:02d}_task-active_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif') for sub_num in range(2, 35))
for act_ep in act_eps:
    try:
        print(act_ep[0])
        sub_active_epochs = mne.read_epochs(act_ep[0], verbose='WARNING')
        act_neg_avgs.append(sub_active_epochs['active/neg'].average())
        act_pos_avgs.append(sub_active_epochs['active/pos'].average())
    except Exception as e:
        print(f"Error: {e}")
        pass

pas_pos_avgs = []
pas_neg_avgs = []
pas_eps = sorted(glob(deriv_dir_status+f'/sub-{sub_num:02d}_task-passive_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif') for sub_num in range(2,35))
for pas_ep in pas_eps:
    try:
        if 'sub-08' in pas_ep[0] or 'sub-10' in pas_ep[0]:
            continue
        print(pas_ep[0])
        sub_passive_epochs = mne.read_epochs(pas_ep[0], verbose='WARNING')
        pas_neg_avgs.append(sub_passive_epochs['passive/neg'].average())
        pas_pos_avgs.append(sub_passive_epochs['passive/pos'].average())
    except Exception as e:
        print(f"Error: {e}")
        pass

motor_avgs = []
motor_eps = sorted(glob(deriv_dir_motor+f'/sub-{sub_num:02d}_task-motor_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif') for sub_num in range(2, 35))
for motor_ep in motor_eps:
    try:
        if 'sub-08' in motor_ep[0] or 'sub-10' in motor_ep[0]:
            continue
        print(motor_ep[0])
        sub_motor_epoch = mne.read_epochs(motor_ep[0], verbose=False)
        motor_avgs.append(sub_motor_epoch.average())
    except Exception as e:
        print(f"Error occurred: {e}")
        pass

In [ ]:
active_avgs = [mne.combine_evoked([act_pos_avgs[x], act_neg_avgs[x]], weights='nave')
                for x in range(len(act_pos_avgs))]
passive_avgs = [mne.combine_evoked([pas_pos_avgs[x], pas_neg_avgs[x]], weights='nave')
                for x in range(len(pas_pos_avgs))]

In [ ]:
picks = 'Cz'  # or a list of channels, or 'eeg'

conditions = {
    'Active': active_avgs ,
    'Passive': passive_avgs,
    'Motor': motor_avgs,
}

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

for cond_name, evokeds in conditions.items():
    # generate one color per subject from a colormap
    colors = cm.Blues(np.linspace(0.3, 0.8, len(evokeds)))
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ch_idx = evokeds[0].ch_names.index('Cz')
    
    for evoked, color in zip(evokeds, colors):
        ax.plot(evoked.times, evoked.get_data()[ch_idx] * 1e6,
                color=color, alpha=0.4, linewidth=0.8)
    
    grand_avg = mne.grand_average(evokeds)
    ax.plot(grand_avg.times, grand_avg.get_data()[ch_idx] * 1e6,
            color='navy', linewidth=2, label='Grand Average')
    
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set(xlabel='Time (s)', ylabel='Amplitude (µV)', title=cond_name)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import mplcursors

for cond_name, evokeds in conditions.items():
    fig, ax = plt.subplots(figsize=(10, 4))
    ch_idx = evokeds[0].ch_names.index('Cz')
    lines = []
    
    for i, evoked in enumerate(evokeds):
        line, = ax.plot(evoked.times, evoked.get_data()[ch_idx] * 1e6,
                        color='steelblue', alpha=0.4, linewidth=0.8)
        line.set_label(f'sub-{i}')  # or parse from evoked.comment
        lines.append(line)
    
    # hover to see subject label
    cursor = mplcursors.cursor(lines, hover=True)
    cursor.connect("add", lambda sel: sel.annotation.set_text(sel.artist.get_label()))
    
    ax.set(xlabel='Time (s)', ylabel='Amplitude (µV)', title=cond_name)
    plt.tight_layout()
    plt.show()

In [ ]:
import plotly.graph_objects as go
import re

sub_nums = np.concatenate([np.arange(1, 8), [9], np.arange(10, 35)])

def plot_condition(evokeds, cond_name, ch_name='Cz'):
    fig = go.Figure()
    ch_idx = evokeds[0].ch_names.index(ch_name)

    for i, evoked in enumerate(evokeds):
        # match = re.search(r'sub-\d+', evoked.comment)
        # label = match.group() if match else 'unknown'
        sub_num = sub_nums[i]
        label = f'sub-{sub_num:02d}'

        fig.add_trace(go.Scatter(
            x=evoked.times,
            y=evoked.get_data()[ch_idx] * 1e6,
            mode='lines',
            name=label,
            line=dict(width=1),
            opacity=0.5,
            hovertemplate=f'<b>{label}</b><br>Time: %{{x:.3f}} s<br>Amplitude: %{{y:.2f}} µV<extra></extra>',
        ))

    # grand average on top
    grand_avg = mne.grand_average(evokeds)
    fig.add_trace(go.Scatter(
        x=grand_avg.times,
        y=grand_avg.get_data()[ch_idx] * 1e6,
        mode='lines',
        name='Grand Average',
        line=dict(width=3, color='red'),
        hovertemplate='<b>Grand Average</b><br>Time: %{x:.3f} s<br>Amplitude: %{y:.2f} µV<extra></extra>',
    ))

    fig.update_layout(
        title=cond_name,
        xaxis_title='Time (s)',
        yaxis_title='Amplitude (µV)',
        hovermode='closest',
        template='plotly_white',
    )
    fig.show()

for cond_name, evokeds in conditions.items():
    plot_condition(evokeds, cond_name)

In [ ]:
pas_pos_grandavg = mne.grand_average(pas_pos_avgs)
pas_neg_grandavg = mne.grand_average(pas_neg_avgs)
passive_grandavg = mne.grand_average(pas_pos_avgs+pas_neg_avgs)

act_pos_grandavg = mne.grand_average(act_pos_avgs)
act_neg_grandavg = mne.grand_average(act_neg_avgs)
active_grandavg = mne.grand_average(act_pos_avgs+act_neg_avgs)

motor_grandavg = mne.grand_average(motor_avgs)

In [ ]:
print(active_grandavg.info['sfreq'] == motor_grandavg.info['sfreq'])
print(active_grandavg.times.shape == motor_grandavg.times.shape)
print(active_grandavg.ch_names == motor_grandavg.ch_names)

subtracted = mne.combine_evoked([active_grandavg, motor_grandavg], weights=[1, -1])
added = mne.combine_evoked([passive_grandavg, motor_grandavg], weights=[1, 1])

evokeds_dict = {
    'Active': active_grandavg,
    'Passive': passive_grandavg,
    'Motor': motor_grandavg,
    'Active - Motor': subtracted,
    'Passive + Motor': added
}

In [ ]:
# all
mne.viz.plot_compare_evokeds(
    evokeds_dict,
    picks='Cz',
    combine='mean',
    legend=True
)

In [ ]:
# subtraction
evokeds_dict = {
    'Active': active_grandavg,
    'Passive': passive_grandavg,
    'Motor': motor_grandavg,
    'Active - Motor': subtracted
}
mne.viz.plot_compare_evokeds(
    evokeds_dict,
    picks='Cz',
    combine='mean',
    legend=True
)

In [ ]:
# added
evokeds_dict = {
    'Active': active_grandavg,
    'Passive': passive_grandavg,
    'Motor': motor_grandavg,
    'Passive + Motor': added
}
mne.viz.plot_compare_evokeds(
    evokeds_dict,
    picks='Cz',
    combine='mean',
    legend=True
)

In [ ]:
active_combined_polarities = act_pos_avgs+act_neg_avgs
evokeds = dict(motor=motor_avgs, active=active_combined_polarities)

In [ ]:
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             picks='Cz', 
                             ci=0.95)

In [ ]:
print(len(active_avgs), len(motor_avgs), len(passive_avgs))
subtracted = []
for i in range(len(active_avgs)):
    subtracted.append(mne.combine_evoked([active_avgs[i], motor_avgs[i]], weights=[1, -1]))
# subtracted = mne.combine_evoked([active_avgs, motor_avgs], weights=[1, -1])

In [ ]:
f_low=4
f_high=30
t_low=0.00
t_high=0.3

# Extract power for passive condition
pas_peak_lat_list = []
pas_peak_amp_list = []
for evk in passive_avgs:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    pas_peak_lat_list.append(peak[1])
    pas_peak_amp_list.append(peak[2])

# Extract power for active condition
act_peak_lat_list = []
act_peak_amp_list = []
for evk in active_avgs:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    act_peak_lat_list.append(peak[1])
    act_peak_amp_list.append(peak[2])


# motor 
mot_peak_lat_list = []
mot_peak_amp_list = []
for evk in subtracted:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    mot_peak_lat_list.append(peak[1])
    mot_peak_amp_list.append(peak[2])

# Create a DataFrame for power values
peak_lat_dict = {'passive': pas_peak_lat_list,  
                 'active': act_peak_lat_list,
                 'motor': mot_peak_lat_list}

peak_amp_dict = {'passive': pas_peak_amp_list,  
                 'active': act_peak_amp_list,
                 'motor': mot_peak_amp_list}
peak_lat_df = pd.DataFrame(peak_lat_dict)
peak_amp_df = pd.DataFrame(peak_amp_dict)

In [ ]:
evokeds = dict(subtracted=subtracted, passive=passive_avgs, active=active_avgs, motor=motor_avgs)
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             picks='Cz', 
                             ci=0.95)

In [ ]:
# Compute the mean peak for each condition
peak_pass_mean = peak_lat_df['passive'].mean()
peak_sub_mean = peak_lat_df['motor'].mean()
print(f'Mean passive peak latency: {peak_pass_mean:.03f} s')
print(f'Mean active - subtracted peak latency: {peak_sub_mean:.03f} s')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_lat_df['passive'], peak_lat_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")

In [ ]:
peak_df = pd.DataFrame(peak_amp_dict)
print(peak_df.head())

# Compute the mean peak for each condition
peak_pass_mean = peak_df['passive'].mean()
peak_sub_mean = peak_df['motor'].mean()
print(f'Mean passive peak amplitude: {peak_pass_mean} ')
print(f'Mean active - subtracted peak amplitude: {peak_sub_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_df['passive'], peak_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")